# Step 8 — A workflow

*Step 8 of the AI in Industry lab*

---

## Read this before you run anything

You will build a fixed pipeline — a flowchart — that reads a student email, extracts fields from it, and routes it.

**What you should end up understanding:** How a model becomes part of ordinary software: it turns prose into JSON that an `if` statement can act on.

| | |
|---|---|
| **Cost** | 4 API calls |
| **Needs earlier steps?** | No. This notebook sets itself up. |
| **Safe to re-run?** | Yes. Each full run of the pipeline costs about 2 calls. |

<div style="background:#fff8e1;border-left:6px solid #f9a825;padding:12px 16px;margin:10px 0;border-radius:4px"><b>Uses about 4 API calls.</b> Re-running cells is fine, it just uses a little more of your free quota each time.</div>

**Now run the Setup cell.** About 30 seconds. It works even if you skipped every earlier step.

In [ ]:
#@title Setup - run this first (about 30 seconds) { display-mode: "form" }
# Fetches the lab files, installs what is needed, reads your API key.
# Identical in every step notebook, so any step works on its own.
import os, sys, pathlib, subprocess

BASE = "/content" if pathlib.Path("/content").exists() else "."
try:
    os.getcwd()
except OSError:
    os.chdir(BASE)
os.chdir(BASE)

if not pathlib.Path("rough").exists():
    print("Downloading the lab files ...")
    subprocess.run("git clone --depth 1 --quiet "
                   "https://github.com/coolMukul/rough.git rough", shell=True)
os.chdir(f"{BASE}/rough/ai-lab")
sys.path.insert(0, os.getcwd())

print("Installing (slow the first time) ...")
subprocess.run(f"{sys.executable} -m pip install -q -r requirements.txt", shell=True)

try:
    from google.colab import userdata
    os.environ["LLM_API_KEY"] = (userdata.get("LLM_API_KEY") or "").strip()
except Exception:
    pass

if not os.environ.get("LLM_API_KEY"):
    print("\n  NO API KEY. Click the key icon on the left, add a secret named")
    print("  exactly LLM_API_KEY, paste your key from console.groq.com/keys,")
    print("  and turn ON 'Notebook access'. Then run this cell again.")
else:
    print(f"\nReady. Key ending ...{os.environ['LLM_API_KEY'][-4:]}")

So far the model answers questions. Real systems **do things** — classify, extract, route, decide.

A workflow is a flowchart. **You have been drawing these since first year.** The only new part is that a model sits inside one of the boxes.

In [ ]:
import json, re
from labcore import chat, corpus, embed, grounded

chunks, texts, _ = corpus()
ask = grounded(embed(texts))

print("ready")

### The incoming message

In [ ]:
# ===== EDIT ME, then run the cell below =====
email = ("Hi, I'm 24BCE0142. I had 68% attendance in Operating Systems "
         "because of placement drives. Will I be allowed to write the end "
         "sem? Please help urgently.")

### Box 1 — the model's only job: turn prose into JSON

An `if` statement cannot read a paragraph. It needs fields.

In [ ]:
def box1_extract(message):
    raw = chat([{"role": "user", "content":
        "Extract fields from this student message. Reply with ONLY a JSON "
        "object and no prose:\n"
        '{"topic": "attendance|grading|degree|other", '
        '"roll_no": "<id or null>", "course": "<name or null>", '
        '"urgent": true|false}\n\n' + message}])
    return json.loads(re.search(r"\{.*\}", raw, re.S).group())


ticket = box1_extract(email)
print(ticket)

**That is structured output.** It is most of the real work in a production pipeline, and it is taught almost nowhere.

### Boxes 2, 3 and 4 — plain Python

No model here at all. Your code decides.

In [ ]:
def handle(message):
    ticket = box1_extract(message)                    # box 1: the model

    if ticket["topic"] == "other":                    # box 2: your code
        return ticket, "ROUTED TO HUMAN - not a regulations question"

    answer = ask(message, k=6)                        # box 3: RAG

    if ticket["urgent"]:                              # box 4: your code
        answer = "[FLAGGED URGENT]\n" + answer

    return ticket, answer


ticket, outcome = handle(email)
print(ticket)
print()
print(outcome[:600])

### Now one that should never reach the answering model

In [ ]:
# ===== EDIT ME, then run the cell below =====
email = "can someone tell me the mess timings for saturday"

In [ ]:
ticket, outcome = handle(email)
print(ticket)
print(outcome)

Classified as `other` and routed to a human — **without ever reaching the RAG step.**

## Why this pattern dominates in industry

Same path every time, so you can test it, cost it, and debug it. **When it breaks, you know which box.**

The price: it only does what you drew. Anything outside the diagram falls straight through.

---

## Now change it yourself

Write your own message in the cell below and watch how it gets classified and routed.

In [ ]:
# ===== EDIT ME, then run the cell below =====
email = "I got 18 out of 60 in my DBMS internals. Am I finished?"

# Things worth trying:
#   - a message with no roll number at all. What does box 2 do?
#   - a very angry message. Does 'urgent' flip to true?
#   - a message about two things at once. Which topic wins?
#   - something in Telugu or Hindi. Does box 1 still return valid JSON?

In [ ]:
ticket, outcome = handle(email)
print(ticket)
print()
print(outcome[:700])

**That last one is worth doing.** Box 1 is the fragile part of every workflow like this — the moment it returns something that is not valid JSON, the whole pipeline throws. Try to break it.

---

### Done with step 8

Open the next step's notebook. If something here did not work, **do not stop to debug it** — every step sets itself up from scratch, so the next one will still run.